# Day 203 — Week 1 Mini-Project: TeleServe India Multi-Turn Triage Agent
### Month 13, Day 203 | LangGraph Fundamentals + Multi-Turn State + Streaming/Parallel/Retries — Combined

**Scenario:** TeleServe India runs a support desk that needs to triage incoming multi-turn customer
conversations in real time. Every customer message must be classified on three axes (urgency, sentiment,
category) **in parallel**, merged into a single priority score, and answered by an agent node — all while
the conversation state survives across turns via checkpointing, streams live, and tolerates transient
classifier failures.

This mini-project is a single integration point for everything built in Week 1:

| Day | Concept | Where it shows up today |
|---|---|---|
| 200 | `StateGraph`, nodes, edges | The whole graph skeleton |
| 201 | Multi-turn state, looping edges, checkpointing | `customer_turn` ↔ `agent_response` loop, `MemorySaver`, `get_state_history` |
| 202 | Streaming, parallel nodes, `RetryPolicy` | Fan-out classifiers, `stream_mode="updates"`, retry behavior |

**Dataset:** seed=203, 10 TeleServe India tickets, 1–3 turns each.
**Stack:** `langgraph==1.2.8`, `langchain-groq==1.1.2` (installed but not required for this notebook's
scored tasks — see the design note in Concept Notes §0 for why).


---
## Setup

In [11]:
!pip install -q langgraph==1.2.8 langchain-groq==1.1.2
print("Setup complete.")

Setup complete.


In [12]:
import operator, json
from typing import TypedDict, Optional, List, Annotated
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import RetryPolicy

print("Imports OK — langgraph pinned to 1.2.8")

Imports OK — langgraph pinned to 1.2.8


---
## Section 1: Raw Data (LOCKED — never modify this cell or its output)

10 TeleServe India support tickets, seed=203. Each ticket is a **scripted multi-turn conversation**
(1–3 customer messages) that the `customer_turn` node will play back one turn per graph cycle.


In [13]:
# LOCKED RAW DATA — seed=203 — do not edit
TICKETS_203 = json.loads(r'''[
  {
    "name": "Rohit Malhotra",
    "tier": "premium",
    "issue": "broadband",
    "turns": [
      "My broadband has been down since this morning, I work from home and this is costing me money.",
      "I already restarted the router twice, it's still not working.",
      "Fine, I'll wait for the technician. Please make sure it's tomorrow morning."
    ],
    "ticket_id": "TS203-01"
  },
  {
    "name": "Ayesha Khan",
    "tier": "standard",
    "issue": "billing",
    "turns": [
      "I was charged twice for my monthly plan this cycle, can you check?",
      "Yes I checked the app, both charges show the same date and amount."
    ],
    "ticket_id": "TS203-02"
  },
  {
    "name": "Vikram Nair",
    "tier": "standard",
    "issue": "mobile_data",
    "turns": [
      "My 4G data speed has been extremely slow for two days.",
      "I'm in Bangalore, Koramangala area.",
      "Okay, I'll try toggling airplane mode and see if that helps."
    ],
    "ticket_id": "TS203-03"
  },
  {
    "name": "Priya Sharma",
    "tier": "premium",
    "issue": "roaming",
    "turns": [
      "I'm traveling to Dubai next week and need international roaming activated urgently.",
      "Can you confirm it will be active before Friday?"
    ],
    "ticket_id": "TS203-04"
  },
  {
    "name": "Sameer Iyer",
    "tier": "standard",
    "issue": "billing",
    "turns": [
      "My bill this month is much higher than usual and I don't understand why.",
      "I did not subscribe to any add-on pack that I'm aware of."
    ],
    "ticket_id": "TS203-05"
  },
  {
    "name": "Neha Kapoor",
    "tier": "premium",
    "issue": "broadband",
    "turns": [
      "The broadband connection keeps disconnecting every 20 minutes.",
      "I've already checked all the cables, they look fine.",
      "Alright, please escalate this since it's affecting my client calls."
    ],
    "ticket_id": "TS203-06"
  },
  {
    "name": "Arjun Desai",
    "tier": "standard",
    "issue": "mobile_data",
    "turns": [
      "I want to know my current data balance for this month."
    ],
    "ticket_id": "TS203-07"
  },
  {
    "name": "Kavita Reddy",
    "tier": "premium",
    "issue": "billing",
    "turns": [
      "I'd like to switch my plan to the annual billing cycle for a discount.",
      "Yes please go ahead and confirm the new amount."
    ],
    "ticket_id": "TS203-08"
  },
  {
    "name": "Manoj Pillai",
    "tier": "standard",
    "issue": "roaming",
    "turns": [
      "My roaming pack expired while I was still abroad and I got charged huge rates.",
      "This is unacceptable, I want a refund for those charges.",
      "Okay, please send me the refund confirmation in writing."
    ],
    "ticket_id": "TS203-09"
  },
  {
    "name": "Divya Menon",
    "tier": "premium",
    "issue": "broadband",
    "turns": [
      "Just wanted to say the technician who came yesterday fixed the issue perfectly, thank you!"
    ],
    "ticket_id": "TS203-10"
  }
]''')

print(f"Loaded {len(TICKETS_203)} tickets")
for t in TICKETS_203:
    print(f"  {t['ticket_id']:10} {t['name']:16} {t['tier']:8} {t['issue']:12} {len(t['turns'])} turn(s)")

Loaded 10 tickets
  TS203-01   Rohit Malhotra   premium  broadband    3 turn(s)
  TS203-02   Ayesha Khan      standard billing      2 turn(s)
  TS203-03   Vikram Nair      standard mobile_data  3 turn(s)
  TS203-04   Priya Sharma     premium  roaming      2 turn(s)
  TS203-05   Sameer Iyer      standard billing      2 turn(s)
  TS203-06   Neha Kapoor      premium  broadband    3 turn(s)
  TS203-07   Arjun Desai      standard mobile_data  1 turn(s)
  TS203-08   Kavita Reddy     premium  billing      2 turn(s)
  TS203-09   Manoj Pillai     standard roaming      3 turn(s)
  TS203-10   Divya Menon      premium  broadband    1 turn(s)


---
## Section 2: Concept Notes

### §0 — Design note: why classifiers are deterministic, not live LLM calls, today
Day 202 already proved the parallel-classifier pattern works with real `ChatGroq` calls
(`openai/gpt-oss-20b`). Today's mini-project is graded on **graph mechanics** — fan-out/fan-in wiring,
multi-turn looping, checkpoint integrity, and retry behavior — not on prompt design, which was already
covered. Using deterministic keyword-rule classifiers keeps every task 100% reproducible and removes API
latency/cost from the loop while you're integrating four days of mechanics at once. A stretch task at the
end shows exactly where to swap in real `ChatGroq` calls using the Day 202 pattern if you want to.

### §1 — The four-node fan-out / one-node fan-in shape (Day 200 + 202)
```
                 ┌─> classify_urgency   ─┐
customer_turn ───┼─> classify_sentiment ─┼──> merge_and_prioritize ──> agent_response ──┐
                 └─> classify_category  ─┘                                              │
        ▲                                                                               │
        └──────────────────────── loop back if NOT resolved ─────────────────────────────┘
```
Three classifier nodes read the *same* upstream state and run concurrently. Because all three also write
to a shared `log` field, `log` **must** carry an `Annotated[List[str], operator.add]` reducer — omitting it
on a key that multiple parallel nodes write raises `InvalidUpdateError` (verified in Day 202; not
re-derived here, but you'll hit it immediately if you forget the reducer in Task 1).

### §2 — Per-turn classification vs. cumulative classification (new distinction for today)
`urgency` and `sentiment` are computed **per turn** (they reflect how the customer feels *right now*,
which is what a live triage system actually needs). `category`, by contrast, is computed on **the full
conversation so far** — a closing reply like *"yes please go ahead"* carries zero category signal on its
own, so category classification must accumulate context across turns while urgency/sentiment intentionally
don't. This is a design decision, not a bug — it's today's interview-framing question (§ Interview Angle).

### §3 — Where the routing decision lives (Day 201 same-step-staleness lesson, applied correctly)
`agent_response` is the node that sets `resolved`. The conditional edge attached to `agent_response`
(`route_conversation`) is what reads `resolved` to decide `end` vs `continue`. Because the edge belongs to
the *same* node that set the flag, LangGraph merges the write before the edge function runs — no staleness.
Contrast this with the Day 201 bug, where `route_after_customer` (attached to `customer_turn`) tried to
read a flag that only `agent_response`, a *different* downstream node, was allowed to set — that's the
shape to avoid.

### §4 — RetryPolicy still only covers transient errors (Day 202 lesson, re-verified in this context)
Attaching `retry_policy=RetryPolicy()` to a classifier node retries `ConnectionError` and similar transient
failures (default 3 attempts) but does **not** retry `ValueError`/`TypeError`/`RuntimeError` — those are
treated as programming errors and propagate immediately. This still held true when verified inside a
fan-out node this session: a `ConnectionError`-raising node succeeded on attempt 3; a `ValueError`-raising
node failed on attempt 1.

### §5 — Streaming shows the fan-out interleaving live
`app.stream(state, config, stream_mode="updates")` emits one update dict per node completion. With three
parallel classifiers, expect to see `classify_urgency`, `classify_sentiment`, and `classify_category`
updates arrive in **non-deterministic order** relative to each other (they're concurrent), but always
*after* `customer_turn` and *before* `merge_and_prioritize` for that turn.


---
## Section 3: Practice Guide

Complete Tasks 1–7 in order. Each task builds on the previous one — by Task 5 you'll run the full graph
across all 10 tickets. Only use LangGraph/Python constructs covered in Days 200–202.


### Task 1 — Define `TicketState` (15 pts)
Define a `TypedDict` called `TicketState` with these fields:
- `messages`: uses `add_messages` reducer (Day 201)
- `turn_count`: `int`
- `resolved`: `bool`
- `category`, `urgency`, `sentiment`, `final_priority`: `Optional[str]` each — **no reducer**, each owned
  by exactly one node
- `log`: `Annotated[List[str], operator.add]` (Day 202) — shared by every node
- `ticket_id`: `str`
- `customer_tier`: `str`
- `script`: `List[str]` — the full turn script for this ticket

**Self-check before moving on:** which fields would raise `InvalidUpdateError` if two nodes wrote to them
in the same step *without* a reducer? Write your answer as a comment.

In [14]:
# ── TASK 1 (15 pts): Define the state schema ─────────────────────────────
# Goal: Create a TypedDict that holds all state needed for the multi‑turn
#       triage agent, with correct reducers for concurrent writes.
# Method:
#   - messages: uses add_messages reducer (Day 201) for appending conversation.
#   - log: uses operator.add reducer (Day 202) because:
#       1. Three classifiers write to it in the same superstep (concurrent).
#       2. Other nodes (customer_turn, merge, agent_response) write to it
#          in later steps (sequential) – without a reducer, each sequential
#          write would overwrite the previous log, losing history.
#   - All other fields are owned by exactly one node, so they are plain types
#     (Optional[str]) – no reducer needed.
#   - If two nodes wrote to the same plain field in the same step, LangGraph
#     would raise InvalidUpdateError – that's why log uses Annotated.

class TicketState(TypedDict):
    messages: Annotated[list, add_messages]   # conversation history
    turn_count: int                           # number of agent replies so far
    resolved: bool                            # True if agent ended with [RESOLVED]
    category: Optional[str]                   # set by classify_category
    urgency: Optional[str]                    # set by classify_urgency
    sentiment: Optional[str]                  # set by classify_sentiment
    final_priority: Optional[str]             # set by merge_and_prioritize
    log: Annotated[List[str], operator.add]   # shared – needs reducer (see above)
    ticket_id: str                            # read‑only after intake
    customer_tier: str                        # read‑only after intake
    script: List[str]                         # full turn script for this ticket

# Self‑check answer (write as a comment):
# Which fields would raise InvalidUpdateError if two nodes wrote to them
# in the same step without a reducer?
# Only `log` – because it is the only field that multiple nodes (the three
# classifiers) write to concurrently in a single superstep. All other fields
# are single‑owner.
#
# Additionally, even without concurrent writes, `log` must have a reducer
# because it is written across multiple sequential steps; without the
# reducer, later writes would overwrite earlier log entries instead of
# appending them.

### Task 2 — `customer_turn` and `agent_response` nodes (15 pts)
- `customer_turn_node`: reads `state["script"][state["turn_count"]]`, appends it as a `human` message,
  appends a log line. Guard against `turn_count >= len(script)`.
- `agent_response_node`: if this is the **last** scripted turn (`turn_count >= len(script) - 1`), reply
  with a message containing the literal tag `[RESOLVED]`; otherwise reply with an acknowledgment that
  references `state["final_priority"]`. Increment `turn_count`. Set `resolved` based on whether
  `[RESOLVED]` is in the reply — parse the tag, don't guess from tone (Day 201 lesson).

In [15]:
# ── TASK 2 (15 pts): customer_turn and agent_response nodes ──────────────
# Goal: Implement the customer turn (plays scripted message) and agent
#       response (acknowledges, increments turn, sets resolved via [RESOLVED]).
# Method:
#   - customer_turn: reads script at index turn_count, appends HumanMessage.
#   - agent_response: if this is the last scripted turn, reply with [RESOLVED];
#     otherwise acknowledge with priority. Parse [RESOLVED] deterministically.

def customer_turn_node(state: TicketState) -> dict:
    """Play the next scripted customer message."""
    idx = state["turn_count"]
    script = state["script"]
    text = script[idx] if idx < len(script) else "(no further input)"
    return {
        "messages": [{"role": "human", "content": text}],
        "log": [f"customer_turn[{idx}]: {text[:40]}"],
    }

def agent_response_node(state: TicketState) -> dict:
    """Generate agent reply, increment turn, set resolved if tag present."""
    idx = state["turn_count"]
    script = state["script"]
    is_last = idx >= len(script) - 1
    if is_last:
        reply = f"Noted, priority {state['final_priority']}. Resolving now. [RESOLVED]"
    else:
        reply = f"Acknowledged (priority {state['final_priority']}), following up..."
    resolved = "[RESOLVED]" in reply
    return {
        "messages": [{"role": "ai", "content": reply}],
        "turn_count": idx + 1,
        "resolved": resolved,
        "log": [f"agent_response[{idx}] resolved={resolved}"],
    }

### Task 3 — Three parallel classifier nodes with `RetryPolicy` (20 pts)
Write `classify_urgency`, `classify_sentiment`, `classify_category` as deterministic keyword-rule
functions (see the keyword lists below — feel free to tune them). Attach `retry_policy=RetryPolicy()` to
all three when you add them to the graph in Task 5.

Keyword lists to use:
```python
URGENCY_KW = ["down", "urgent", "immediately", "asap", "wait", "expired", "disconnect"]
FRUSTRATION_KW = ["already", "costing", "unacceptable", "twice", "don't understand", "not aware"]
CATEGORY_MAP = {
    "broadband": ["broadband", "router", "connection", "technician"],
    "billing": ["charged", "bill", "billing", "refund", "amount", "plan", "discount"],
    "mobile_data": ["data", "4g", "speed", "balance"],
    "roaming": ["roaming", "international", "abroad", "dubai"],
}
```
Remember §2: `urgency`/`sentiment` classify the **latest** message only. `category` classifies **all human
messages so far, joined together** — get this wrong and every multi-turn ticket's category will drift on
confirmation replies.

In [16]:
# ── TASK 3 (20 pts): Three parallel classifier nodes ─────────────────────
# Goal: Deterministic keyword‑based classification with correct scope.
# Method:
#   - urgency and sentiment operate on the *latest* message only (per‑turn).
#   - category uses the *full* human‑message history (cumulative).
#   - All three will get RetryPolicy attached in Task 5.

URGENCY_KW = ["down", "urgent", "immediately", "asap", "wait", "expired", "disconnect"]
FRUSTRATION_KW = ["already", "costing", "unacceptable", "twice", "don't understand", "not aware"]
CATEGORY_MAP = {
    "broadband": ["broadband", "router", "connection", "technician"],
    "billing": ["charged", "bill", "billing", "refund", "amount", "plan", "discount"],
    "mobile_data": ["data", "4g", "speed", "balance"],
    "roaming": ["roaming", "international", "abroad", "dubai"],
}

def classify_urgency(state: TicketState) -> dict:
    """Classify urgency from the latest customer message."""
    last = state["messages"][-1].content.lower()
    val = "high" if any(k in last for k in URGENCY_KW) else "medium"
    return {"urgency": val, "log": [f"classify_urgency -> {val}"]}

def classify_sentiment(state: TicketState) -> dict:
    """Classify sentiment from the latest customer message."""
    last = state["messages"][-1].content.lower()
    val = "frustrated" if any(k in last for k in FRUSTRATION_KW) else "neutral"
    if "thank you" in last or "perfectly" in last:
        val = "positive"
    return {"sentiment": val, "log": [f"classify_sentiment -> {val}"]}

def classify_category(state: TicketState) -> dict:
    """Classify category from the full human‑message history."""
    # Join all human messages so far (cumulative)
    full_text = " ".join(m.content.lower() for m in state["messages"] if m.type == "human")
    best, best_hits = "general", 0
    for cat, kws in CATEGORY_MAP.items():
        hits = sum(1 for k in kws if k in full_text)
        if hits > best_hits:
            best, best_hits = cat, hits
    return {"category": best, "log": [f"classify_category -> {best}"]}

### Task 4 — `merge_and_prioritize` fan-in node (15 pts)
Deterministic NRA-style priority rule:
- `urgency == "high"` **and** `sentiment == "frustrated"` → `"P1"`
- `urgency == "high"` **or** `sentiment == "frustrated"` (but not both) → `"P2"`
- otherwise → `"P3"`
- **Tier boost:** if `customer_tier == "premium"` and the computed priority is `"P2"`, bump it to `"P1"`

In [17]:
# ── TASK 4 (15 pts): Merge and routing logic ─────────────────────────────
# Goal: Compute final_priority with an NRA‑style rule and route based on
#       resolved flag.
# Method:
#   - urgency=high AND sentiment=frustrated → P1
#   - only one of urgency=high or sentiment=frustrated → P2
#   - otherwise → P3
#   - if tier=premium and priority=P2 → bump to P1
#   - routing: END if resolved, else continue to customer_turn.

def merge_and_prioritize(state: TicketState) -> dict:
    """Fan‑in node that combines classifier outputs into a priority."""
    urgency, sentiment, tier = state["urgency"], state["sentiment"], state["customer_tier"]
    if urgency == "high" and sentiment == "frustrated":
        priority = "P1"
    elif urgency == "high" or sentiment == "frustrated":
        priority = "P2"
    else:
        priority = "P3"
    if tier == "premium" and priority == "P2":
        priority = "P1"
    return {
        "final_priority": priority,
        "log": [f"merge -> {priority} (urg={urgency},sent={sentiment},tier={tier})"],
    }

def route_conversation(state: TicketState) -> str:
    """Return next node name based on resolved flag."""
    return "end" if state["resolved"] else "continue"

### Task 5 — Wire the graph and run all 10 tickets (20 pts)
1. Build the `StateGraph(TicketState)`: `customer_turn` fans out to the three classifiers, all three feed
   into `merge_and_prioritize`, which feeds `agent_response`, which conditionally routes back to
   `customer_turn` or to `END`.
2. Compile with a `MemorySaver()` checkpointer.
3. For each of the 10 tickets, run `app.stream(init_state, config, stream_mode="updates")` with
   `thread_id = ticket["ticket_id"]`.
4. For each ticket, print: `ticket_id, name, tier, n_turns, final category/urgency/sentiment/priority,
   number of checkpoints, and a monotonic turn_count check via get_state_history`.

In [18]:
# ── TASK 5 (20 pts): Full graph wiring and batch streaming ──────────────
# Goal: Build the graph with fan‑out/fan‑in, attach RetryPolicy to classifiers,
#       compile with MemorySaver, stream all tickets, and print audit info.
# Method:
#   - customer_turn → three classifiers → merge → agent_response → (conditional back)
#   - Use stream_mode="updates" to show per‑node progress.
#   - For each ticket, print checkpoint history and monotonic turn_count check.

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# Build graph
graph = StateGraph(TicketState)

graph.add_node("customer_turn", customer_turn_node)
graph.add_node("classify_urgency", classify_urgency, retry_policy=RetryPolicy())
graph.add_node("classify_sentiment", classify_sentiment, retry_policy=RetryPolicy())
graph.add_node("classify_category", classify_category, retry_policy=RetryPolicy())
graph.add_node("merge_and_prioritize", merge_and_prioritize)
graph.add_node("agent_response", agent_response_node)

graph.set_entry_point("customer_turn")

# Fan‑out from customer_turn to classifiers
for c in ["classify_urgency", "classify_sentiment", "classify_category"]:
    graph.add_edge("customer_turn", c)
    graph.add_edge(c, "merge_and_prioritize")

# Fan‑in to merge, then agent, then conditional back
graph.add_edge("merge_and_prioritize", "agent_response")
graph.add_conditional_edges("agent_response", route_conversation, {
    "end": END,
    "continue": "customer_turn"
})

# Compile with MemorySaver for checkpointing
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

results = []
for t in TICKETS_203:
    config = {"configurable": {"thread_id": t["ticket_id"]}}
    init = {
        "messages": [],
        "turn_count": 0,
        "resolved": False,
        "category": None,
        "urgency": None,
        "sentiment": None,
        "final_priority": None,
        "log": [],
        "ticket_id": t["ticket_id"],
        "customer_tier": t["tier"],
        "script": t["turns"],
    }
    print(f"\n--- {t['ticket_id']} ({t['name']}) streaming ---")
    # Stream updates – proof of parallel execution
    for update in app.stream(init, config=config, stream_mode="updates"):
        print(update)

    # Retrieve final state and history for audit
    final_state = app.get_state(config).values
    history = list(app.get_state_history(config))
    # Reverse history to chronological, then extract turn_count values
    turn_counts = [
        h.values.get("turn_count") for h in reversed(history)
        if h.values.get("turn_count") is not None
    ]
    monotonic = all(turn_counts[i] <= turn_counts[i+1] for i in range(len(turn_counts)-1))

    results.append({
        "ticket_id": t["ticket_id"],
        "name": t["name"],
        "tier": t["tier"],
        "n_turns": len(t["turns"]),
        "final_category": final_state["category"],
        "final_urgency": final_state["urgency"],
        "final_sentiment": final_state["sentiment"],
        "final_priority": final_state["final_priority"],
        "checkpoints": len(history),
        "monotonic_turn_count": monotonic,
        "resolved": final_state["resolved"],
    })

# Print summary table
print(f"\n{'ticket':10} {'name':16} {'tier':8} {'turns':5} {'cat':12} "
      f"{'urg':6} {'sent':10} {'prio':5} {'ckpts':6} {'monotonic':9}")
for r in results:
    print(f"{r['ticket_id']:10} {r['name']:16} {r['tier']:8} {r['n_turns']:5} "
          f"{r['final_category']:12} {r['final_urgency']:6} {r['final_sentiment']:10} "
          f"{r['final_priority']:5} {r['checkpoints']:6} {str(r['monotonic_turn_count']):9}")


--- TS203-01 (Rohit Malhotra) streaming ---
{'customer_turn': {'messages': [{'role': 'human', 'content': 'My broadband has been down since this morning, I work from home and this is costing me money.'}], 'log': ['customer_turn[0]: My broadband has been down since this mo']}}
{'classify_category': {'category': 'broadband', 'log': ['classify_category -> broadband']}}
{'classify_urgency': {'urgency': 'high', 'log': ['classify_urgency -> high']}}
{'classify_sentiment': {'sentiment': 'frustrated', 'log': ['classify_sentiment -> frustrated']}}
{'merge_and_prioritize': {'final_priority': 'P1', 'log': ['merge -> P1 (urg=high,sent=frustrated,tier=premium)']}}
{'agent_response': {'messages': [{'role': 'ai', 'content': 'Acknowledged (priority P1), following up...'}], 'turn_count': 1, 'resolved': False, 'log': ['agent_response[0] resolved=False']}}
{'customer_turn': {'messages': [{'role': 'human', 'content': "I already restarted the router twice, it's still not working."}], 'log': ['customer_turn

### Task 6 — Prove the RetryPolicy behavior inside this graph shape (10 pts)
Build two tiny **separate** one-node graphs (don't touch the main graph above):
1. A node that raises `ConnectionError` on its first two calls (use a module-level counter) and succeeds
   on the third, with `retry_policy=RetryPolicy()` attached. Invoke it and print the result + attempt count.
2. A node that always raises `ValueError`, with `retry_policy=RetryPolicy()` attached. Invoke it inside a
   `try/except` and print the attempt count when it fails.

Print a one-line conclusion stating which exception type retried and which didn't, and why that matters
for a classifier node calling a real LLM API in production.

In [19]:
# ── TASK 6 (10 pts): Isolated RetryPolicy proof ──────────────────────────
# Goal: Demonstrate that RetryPolicy retries ConnectionError but not ValueError.
# Method:
#   - Build two small graphs: one with a node that raises ConnectionError twice,
#     then succeeds; another that raises ValueError immediately.
#   - Print attempt counts to prove the different behaviours.

from typing import TypedDict, Optional

class TinyState(TypedDict):
    status: Optional[str]

# Node that fails with ConnectionError twice, then succeeds
attempt_counter = {"conn": 0, "val": 0}

def flaky_connection_node(state):
    attempt_counter["conn"] += 1
    if attempt_counter["conn"] < 3:
        raise ConnectionError(f"transient network error, attempt {attempt_counter['conn']}")
    return {"status": f"succeeded on attempt {attempt_counter['conn']}"}

def flaky_value_node(state):
    attempt_counter["val"] += 1
    raise ValueError(f"bad classifier output, attempt {attempt_counter['val']}")

# Graph 1: ConnectionError – should retry and succeed on attempt 3
g1 = StateGraph(TinyState)
g1.add_node("flaky", flaky_connection_node, retry_policy=RetryPolicy())
g1.set_entry_point("flaky")
g1.add_edge("flaky", END)
app1 = g1.compile()
result1 = app1.invoke({"status": None})
print("ConnectionError node result:", result1, "| attempts:", attempt_counter["conn"])

# Graph 2: ValueError – should fail immediately (no retry)
g2 = StateGraph(TinyState)
g2.add_node("flaky", flaky_value_node, retry_policy=RetryPolicy())
g2.set_entry_point("flaky")
g2.add_edge("flaky", END)
app2 = g2.compile()
try:
    app2.invoke({"status": None})
    print("UNEXPECTED: ValueError node did not raise")
except ValueError as e:
    print("ValueError node raised immediately (no retry), attempts:", attempt_counter["val"], "| error:", e)

# One‑line conclusion
print("CONCLUSION: ConnectionError is retried (3 attempts), ValueError is not retried (1 attempt). "
      "In production, any custom ValueError raised by a classifier (e.g., unparseable output) "
      "must be handled explicitly, or use a custom retry_on to retry it.")

ConnectionError node result: {'status': 'succeeded on attempt 3'} | attempts: 3
ValueError node raised immediately (no retry), attempts: 1 | error: bad classifier output, attempt 1
CONCLUSION: ConnectionError is retried (3 attempts), ValueError is not retried (1 attempt). In production, any custom ValueError raised by a classifier (e.g., unparseable output) must be handled explicitly, or use a custom retry_on to retry it.


### Task 7 — NRA summary + interview framing (5 pts)
Write one NRA block (Number + Reason + Action) summarizing the triage run across all 10 tickets, and one
interview-framing paragraph answering: *"Why did you classify category cumulatively but urgency/sentiment
per-turn — and where would this design break down at scale?"*

In [20]:
# ── TASK 7 (5 pts): NRA summary and interview framing ────────────────────
# Write your NRA block and interview framing text as comments in this cell.

# NRA:
# Number: Out of 10 tickets, only TS203-01 received a P1 priority; the other 9
#         were assigned P3. All tickets resolved successfully (resolved=True),
#         and all had monotonic turn_count progression (True).
# Reason: The priority rule depends on the *latest* turn's urgency and sentiment.
#         Most tickets' final turn is a calm acknowledgment or confirmation,
#         so they lose the high‑urgency/frustration signal that existed earlier.
#         Only TS203-01 still contained an urgency keyword ("wait") on its final
#         turn, triggering P1. The category classifier, using cumulative history,
#         correctly assigned broadband/billing/mobile_data/roaming across all tickets.
# Action: Since P1 tickets are the most critical, I would adjust the priority
#         rule to consider the *maximum* urgency/sentiment seen across all turns,
#         or introduce a separate "escalation threshold" based on persistent
#         frustration. I would also log a warning whenever a ticket's final
#         priority drops from P1 to P3, as an alert for manual review.

# Interview framing (answer to: "Why did you classify category cumulatively but
# urgency/sentiment per‑turn – and where would this design break down at scale?")
#
# "I chose per‑turn urgency/sentiment because a customer's emotional state
# changes over a conversation – they might start frustrated and become calmer
# after receiving a solution. For a live triage system, the most recent
# sentiment is what the agent needs to respond to *right now*. In contrast,
# category is a property of the *issue itself*, not the mood. A closing reply
# like 'yes please go ahead' carries no category signal, so classifying it in
# isolation would corrupt the label. By accumulating all human messages so far,
# category classification remains stable and accurate even on short confirmations.
#
# At scale, this design breaks down when a customer's final turn is neutral but
# the earlier turns were extremely urgent – we would under‑prioritise them.
# To fix that, we would either use the maximum urgency across all turns as a
# separate 'severity' score, or introduce a sliding window that decays older
# urgency signals. The right choice depends on the business SLA: if an issue
# that was urgent once still matters hours later, we should accumulate urgency;
# if urgency is truly moment‑to‑moment, per‑turn is correct."

---
## Section 4: Scoring Rubric (100 pts)

| Task | Points | Criteria |
|---|---|---|
| Task 1 — `TicketState` schema | 15 | Correct reducers on `messages` and `log`; no reducer on single-owner fields; correct self-check answer |
| Task 2 — `customer_turn` / `agent_response` | 15 | Correct message roles (`human`/`ai`); `[RESOLVED]` tag parsed, not inferred from tone; `turn_count` incremented in the right node |
| Task 3 — three classifiers | 20 | `urgency`/`sentiment` use latest message; `category` uses **full** human-message history; correct keyword logic |
| Task 4 — `merge_and_prioritize` | 15 | Correct P1/P2/P3 rule; premium tier boost applied only to P2→P1, not P3→P2 |
| Task 5 — full graph + 10-ticket run | 20 | Correct fan-out/fan-in wiring; `RetryPolicy()` attached to all 3 classifiers; streaming used (not `.invoke()`); checkpoint/monotonic check printed per ticket |
| Task 6 — RetryPolicy proof | 10 | Both flaky nodes built correctly; correct attempt counts; correct one-line conclusion |
| Task 7 — NRA + interview framing | 5 | NRA has no Reason→Action bleed; interview answer directly addresses the per-turn vs. cumulative design tradeoff |

**Technical vs. Communication split:** Tasks 1–6 are scored as technical (correct code, correct verified
output). Task 7 and any prose in Task 5/6 conclusions are scored as communication — a correct
`final_priority` table with a vague or hedged NRA still loses Task 7 points.

**Key Takeaway:** the hardest part of this mini-project isn't any single Week 1 concept — it's deciding
*which state fields need reducers and which don't* when four different node types (turn-taking, parallel
classifiers, fan-in, retry-wrapped) all touch the same state object at once.
